In [ ]:
from mxdataspark import MXData
from mxdataspark.core import MXDataClient


In [ ]:
from importlib.metadata import version
print(f'MXData Spark package version: {version("mxdataspark")}')


In [ ]:
# DBTITLE 1,Widget Parameters
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("load_group", "-1")
dbutils.widgets.text("source_id", "-1")
dbutils.widgets.text("flow_id", "-1")
dbutils.widgets.text("job_concurrency", "")


In [ ]:
load_group = int(dbutils.widgets.get("load_group"))
source_id = int(dbutils.widgets.get("source_id"))
flow_id = dbutils.widgets.get("flow_id")
job_concurrency = dbutils.widgets.get("job_concurrency")
job_concurrency = int(job_concurrency) if int(job_concurrency or 0) > 0 else None

catalog_name = dbutils.widgets.get("catalog_name").lower()
spark.catalog.setCurrentCatalog(catalog_name)


In [ ]:
client = MXDataClient()
ingest_configs = client.get_ingestion_object_list(
    load_group=load_group, source_id=source_id, flow_id=flow_id
)

print("MXData.queue_ingestion_batch Configs:")
print(f"{ingest_configs=}")
print(f"{job_concurrency=}")


In [ ]:
# DBTITLE 1,Main
metrics = (
    MXData.engine("batch")
    .queue_ingestion_batch(ingest_configs)
    .process_batch(workers=job_concurrency)
    .metrics("dict")
)


In [ ]:
FAIL = False

for metric in metrics:
    if not metric["success"]:
        FAIL = True


In [ ]:
if FAIL:
    print(f"An ingestion has failed! Check the Metrics.")
    for metric in metrics:
        if not metric["success"]:
            print(metric)
    raise ValueError("An ingestion has failed! Check ObjectIngest. Exiting... ")
